# Extra questions: DocLens + gated visual Adjudicator on 400 NEW MMLongBench-Doc questions (Qwen3-VL-8B)

Extends the evaluation beyond the 200 questions of the DocLens paper's Table 9 sample. It draws **400 other questions** (never in the first 200,
seed 1) and splits them into `NUM_PARTS` equal parts so that **several Kaggle accounts run in parallel**; each account sets its own `PART_ID`.

For its part, this notebook runs, in order:
1. **DocLens** (Page Navigator -> Answer Sampler -> text-only Adjudicator, k = 8) with scoring of the 8 sampled answers;
2. the **gated visual Adjudicator** (all 8 answers agree -> no Adjudicator; otherwise the Adjudicator also sees the evidence pages), reusing step 1.

Both write `results_extra/MMLongBenchDoc/extra_doclens_p<PART_ID>.jsonl` and `extra_visual_adj_p<PART_ID>.jsonl`. At the end every part shows its own report.

**How to run (every account, same steps)**
1. *Settings*: **GPU T4 x2**, **Internet On**. Check *Your Kaggle Quota* first (a 50-question part needs about 6 hours).
2. Set `PART_ID` in the settings cell (account 1 -> 1, account 2 -> 2, ... up to `NUM_PARTS`).
3. *Add Input* -> *Your Work* -> **Agentic** (the version with `parsed_MMLongBenchDoc.tar`).
   To continue an unfinished part, also add **that part's own latest version** as an input; its saved answers are picked up.
4. **Save Version -> "Save & Run All (Commit)" -> Save.**
5. When *Complete*: download `results_extra/MMLongBenchDoc/extra_doclens_p<PART_ID>.jsonl`, `extra_visual_adj_p<PART_ID>.jsonl`
   and `visual_adj_report.md` from *Output*.


In [ ]:
# ======================= SETTINGS =======================
import time
T_START = time.time()
TIME_BUDGET_H = 11.0              # stop the runs after this many hours (Kaggle ends every session at 12)
WORK = "/kaggle/working"
CDC_REPO = "https://github.com/Mariam1543/aqqal123.git"
CDC_BRANCH = "claude/wizardly-rubin-kdthhy"
DATASET = "MMLongBenchDoc"

N_BASE = 200                      # the questions already evaluated (Table 9 sample, seed 0): these are EXCLUDED here
N_EXTRA = 400                     # new questions drawn from the rest of the benchmark (seed 1)
NUM_PARTS = 8                     # the 400 are split into this many equal parts (50 questions each)
PART_ID = 1                       # THIS account's part: 1 ... NUM_PARTS (a different number in every account)
SEED = 0
RUN_ONLY = ["qwen_doclens"]      # its settings (model, k = 8) are used for both steps

MAX_MODEL_LEN = 40960             # context that fits next to the 8B weights on 2x T4
NAV_CHUNK = 6                     # Page Navigator pages per request (short prompts: attention on T4 grows with length^2)
QWEN_PARALLEL = 4                 # Qwen requests on the server at once (each carries all 8 samples)
MAX_IMAGES = 48                   # images per request accepted by the server
MAX_CONCURRENT = 4
PAPER_PROMPTS = True
PILOT_QUESTIONS = 0               # 0 = no pilot (already measured: ~4.5 min per DocLens question)

## 1. Get the code

In [ ]:
import os, sys, subprocess, shutil, json, glob, random, time
os.chdir(WORK)
if not os.path.exists("DocLens"):
    subprocess.run(["git", "clone", "https://github.com/dwzhu-pku/DocLens.git"], check=True)
shutil.rmtree("doclens_cdc", ignore_errors=True)   # always take the latest version of the extension
shutil.rmtree("_cdc_repo", ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "-b", CDC_BRANCH, CDC_REPO, "_cdc_repo"], check=True)
shutil.copytree("_cdc_repo/doclens_cdc", "doclens_cdc")
shutil.rmtree("_cdc_repo")
DOCLENS, CDC = f"{WORK}/DocLens", f"{WORK}/doclens_cdc"
ROOT = f"{WORK}/full_{DATASET}"
RESULTS = f"{WORK}/results_extra/{DATASET}"
os.makedirs(RESULTS, exist_ok=True)
sys.path.insert(0, CDC)
from reproduction.experiments import EXPERIMENTS, run_args
SELECTED = {k: EXPERIMENTS[k] for k in RUN_ONLY}
for k, e in SELECTED.items():
    print(f"{k:14s} {e['note']:40s} K={e['k1']}/{e['k2']}")

## 2. Install vLLM and the DocLens dependencies (about 10 minutes)

In [ ]:
!pip install -q "vllm>=0.11.0" 2>&1 | tail -n 3
!pip install -q -r {CDC}/requirements-kaggle.txt 2>&1 | tail -n 2
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| GPUs', torch.cuda.device_count())"

## 3. Restore the MinerU results and re-create the page images

In [ ]:
tars = glob.glob("/kaggle/input/**/parsed_MMLongBenchDoc.tar", recursive=True)
assert tars, "Add the saved notebook version containing parsed_MMLongBenchDoc.tar as an input (right panel: Add Input)."
if not os.path.exists(f"{ROOT}/clean/data/{DATASET}/documents"):
    !tar -xf {tars[0]} -C {WORK}
!python {CDC}/scripts/download_data.py --dataset {DATASET} --out {DOCLENS}/data
SAMPLES = f"{DOCLENS}/data/{DATASET}/samples.json"
DOCUMENTS = f"{DOCLENS}/data/{DATASET}/documents"
# The tar holds MinerU's output but not the page screenshots; re-render them (skips existing ones).
!python {CDC}/scripts/prepare_subset.py --all --samples {SAMPLES} --documents {DOCUMENTS} --out_root {ROOT} --dataset_name {DATASET} --workers 4
docs = f"{ROOT}/clean/data/{DATASET}/documents"

def qkey(r):
    return (r["doc_id"], r["question"], str(r.get("answer", "")))

def n_done(path, want=None):
    # finished questions in a results file (failed attempts and repeats not counted; only `want` if given)
    if not os.path.exists(path):
        return 0
    keys = set()
    for line in open(path, encoding="utf-8"):
        if line.strip():
            r = json.loads(line)
            if "cdc_error" not in r:
                keys.add(qkey(r))
    return len(keys & want) if want is not None else len(keys)

# Resume: add the finished questions of THIS part found in attached earlier versions of this notebook
for prev in sorted(glob.glob(f"/kaggle/input/**/results_extra/{DATASET}/*_p{PART_ID}.jsonl", recursive=True)):
    dst = f"{RESULTS}/{os.path.basename(prev)}"
    have = set()
    if os.path.exists(dst):
        have = {qkey(json.loads(l)) for l in open(dst, encoding="utf-8") if l.strip() and "cdc_error" not in json.loads(l)}
    added = 0
    with open(dst, "a", encoding="utf-8") as f:
        for line in open(prev, encoding="utf-8"):
            if not line.strip():
                continue
            r = json.loads(line)
            if "cdc_error" in r or qkey(r) in have:
                continue
            f.write(line.rstrip("\n") + "\n"); have.add(qkey(r)); added += 1
    print(f"Merged {os.path.basename(prev)} from {os.path.dirname(prev)}: +{added} questions")
print(len(glob.glob(f"{docs}/*/*.jpeg")), "page images,", len(glob.glob(f"{docs}/*/MinerU_Page*/*.md")), "parsed pages (expected 6529 each)")


## 3b. This account's part of the 400 new questions

In [ ]:
# The 200 questions already evaluated (same seeded draw as before) are excluded; the 400 new ones are a seeded draw
# from the remaining questions, split into NUM_PARTS parts. This account runs part PART_ID.
all_q = json.load(open(SAMPLES))
base = random.Random(SEED).sample(range(len(all_q)), N_BASE)
rest = [i for i in range(len(all_q)) if i not in set(base)]
extra = random.Random(1).sample(rest, N_EXTRA)
assert not set(base) & set(extra) and len(set(extra)) == N_EXTRA
assert 1 <= PART_ID <= NUM_PARTS, "PART_ID must be between 1 and NUM_PARTS"
size = N_EXTRA // NUM_PARTS
mine = extra[(PART_ID - 1) * size : (PART_ID * size if PART_ID < NUM_PARTS else N_EXTRA)]
picked = set(mine)
N_QUESTIONS = len(picked)
QUESTIONS = f"{ROOT}/samples_extra_part{PART_ID}of{NUM_PARTS}.json"
json.dump([q for i, q in enumerate(all_q) if i in picked], open(QUESTIONS, "w"), ensure_ascii=False, indent=1)
json.dump({"base": base, "extra": extra, "this_part": mine}, open(f"{RESULTS}/extra_indices_p{PART_ID}.json", "w"))
WANT = {(q["doc_id"], q["question"], str(q.get("answer", ""))) for i, q in enumerate(all_q) if i in picked}
print(f"part {PART_ID}/{NUM_PARTS}: {N_QUESTIONS} new questions from {len({all_q[i]['doc_id'] for i in picked})} documents -> {QUESTIONS}")


## 4. Start the Qwen3-VL-8B server (downloads ~17 GB the first time, then loads onto both T4s)

In [ ]:
import urllib.request
log = open(f"{WORK}/vllm.log", "w")
server = subprocess.Popen([
    "vllm", "serve", "Qwen/Qwen3-VL-8B-Instruct",
    "--tensor-parallel-size", "2", "--dtype", "half",
    "--max-model-len", str(MAX_MODEL_LEN), "--gpu-memory-utilization", "0.92",
    "--limit-mm-per-prompt", json.dumps({"image": MAX_IMAGES}),
    "--mm-processor-kwargs", json.dumps({"max_pixels": 1003520}),
    "--max-num-seqs", "32", "--enable-prefix-caching", "--enforce-eager",
    "--disable-custom-all-reduce", "--port", "8000",
], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)  # survives a Stop/Interrupt

start = time.time()
while True:
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
        print(f"Server ready after {(time.time() - start) / 60:.1f} min"); break
    except Exception:
        if server.poll() is not None or time.time() - start > 40 * 60:
            print(open(f"{WORK}/vllm.log").read()[-6000:])
            raise RuntimeError("vLLM did not start: send the log above.")
        time.sleep(20)

In [ ]:
# Smoke test: one real question about one real page
import base64, openai
client = openai.OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")
page = sorted(glob.glob(f"{docs}/*/1.jpeg"))[0]
b64 = base64.b64encode(open(page, "rb").read()).decode()
r = client.chat.completions.create(model="Qwen/Qwen3-VL-8B-Instruct", max_tokens=60, messages=[{"role": "user", "content": [
    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
    {"type": "text", "text": "What is the title of this page?"}]}])
print(page, "->", r.choices[0].message.content)

## 5. Run settings

In [ ]:
common = (f"python {CDC}/run_cdc.py --doclens_root {DOCLENS} --data_root {ROOT}/clean --variant clean "
          f"--dataset_name {DATASET} --samples {QUESTIONS} --max_concurrent {MAX_CONCURRENT} "
          f"--qwen_base_url http://localhost:8000/v1 --qwen_max_images {MAX_IMAGES} --nav_chunk_pages {NAV_CHUNK} --qwen_parallel_requests {QWEN_PARALLEL} "
          f"{'--paper_prompts' if PAPER_PROMPTS else ''}")


## 6. Run: DocLens, then the gated visual Adjudicator
A progress line is printed every 10 minutes. Logs: `extra_doclens_p<PART_ID>.log`, `extra_visual_adj_p<PART_ID>.log`.

In [ ]:
import signal
DOC_OUT = f"{RESULTS}/extra_doclens_p{PART_ID}.jsonl"
ADJ_OUT = f"{RESULTS}/extra_visual_adj_p{PART_ID}.jsonl"
base_args = run_args(SELECTED["qwen_doclens"])
STEPS = [
    ("DocLens (8 samples + text-only Adjudicator + scoring)", f"{common} {base_args} --output {DOC_OUT}", DOC_OUT),
    ("Gated visual Adjudicator (reuses the DocLens answers above)",
     f"{common} {base_args} --mode visual_adj --reuse_from {DOC_OUT} --output {ADJ_OUT}", ADJ_OUT),
]
def budget_left():
    return T_START + TIME_BUDGET_H * 3600 - time.time()

for title, cmd, out in STEPS:
    for attempt in range(3):          # extra passes retry questions whose call failed
        if n_done(out, WANT) >= N_QUESTIONS:
            break
        if budget_left() < 10 * 60:
            print("Time budget reached; add this version as an input and run again to continue."); break
        print(f"=== {title} | pass {attempt + 1}: {n_done(out, WANT)}/{N_QUESTIONS} done, {budget_left() / 3600:.1f} h left", flush=True)
        log = open(out.replace(".jsonl", ".log"), "a")
        proc = subprocess.Popen(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        t_last = time.time()
        while proc.poll() is None:
            time.sleep(30)
            if time.time() - t_last > 600:
                t_last = time.time()
                print(f"  {time.strftime('%H:%M')}: {n_done(out, WANT)}/{N_QUESTIONS} done", flush=True)
            if budget_left() <= 0:
                os.killpg(proc.pid, signal.SIGTERM); proc.wait()
                print("  stopped at the time budget (saved)", flush=True); break
        log.close()
    print(f"{title}: {n_done(out, WANT)}/{N_QUESTIONS} done")
    if n_done(out, WANT) < N_QUESTIONS:
        break                          # do not start the next step on an unfinished one


## 7. Report of this part

In [ ]:
from IPython.display import Markdown, display
if os.path.exists(ADJ_OUT):
    !python {CDC}/scripts/report_visual_adj.py {ADJ_OUT} --out {RESULTS}/visual_adj_report.md
    if os.path.exists(f"{RESULTS}/visual_adj_report.md"):
        display(Markdown(open(f"{RESULTS}/visual_adj_report.md").read()))


## 8. Clean up so the output saves quickly

In [ ]:
subprocess.run(["pkill", "-f", "vllm serve"])
!rm -rf {ROOT} {DOCLENS}/data
!du -sh {WORK}
for f in sorted(glob.glob(f"{RESULTS}/*.jsonl")):
    print(os.path.basename(f), n_done(f, WANT), "of this part's", N_QUESTIONS, "questions")
